Every standard collection in Python is iterable. An iterable is an object that provides
an iterator, which Python uses to support operations like:
• for loops
• List, dict, and set comprehensions
• Unpacking assignments
• Construction of collection instances

In [4]:
import random
def roll_dice():
    return random.randint(1, 6)

for roll in iter(roll_dice, 6):
    print(roll)
print("got a 6!")




4
got a 6!


Iterable (The Data): A collection of items. It doesn't know anything about "where" you are in the collection; it just holds the data. Examples include strings ('ABC'), lists ([1, 2, 3]), and dictionaries.

Iterator (The Pointer): An object with a memory of its current state. It doesn't necessarily hold the data; instead, it points to a specific location within the iterable and knows how to fetch the next item.

In [10]:
m = iter('abc')
while True:
    try:
        print(next(m))
    except StopIteration:
        print("No more items!")
        del m
        break






a
b
c
No more items!


In [11]:
m = list(iter('abc'))
m


['a', 'b', 'c']

In [23]:
data = [1, 2, 3, 4, 6]

for i in data:
    if i % 2 == 0:
        data.remove(i)

print(data)
dataa = [1, 2, 3, 4, 6]
dataa = [i for i in dataa if i%2 != 0]
dataa

[1, 3, 6]


[1, 3]

a generator does not return values in the normal sense; it yields them.

Think of yield as a pause button. When Python hits a yield statement, it packages up the value, hands it back to the caller, and then freezes the function exactly as it is

In [26]:
def gen_squares(n):
    yield n*4

print(gen_squares(4))
next(gen_squares(4))

<generator object gen_squares at 0x0000026AB860E080>


16

In [27]:
def find_first_error_eager(file_path):
    # EAGER: Reads all 10 million lines into a massive list in memory first!
    with open(file_path, 'r') as file:
        all_lines = file.readlines() 
        
    # Now loop through the giant list in memory
    for line in all_lines:
        if "ERROR" in line:
            return line

In [28]:
def find_first_error_lazy(file_path):
    with open(file_path, 'r') as file:
        # LAZY: The file object itself is a generator. 
        # It yields one line at a time from the hard drive, then pauses.
        for line in file: 
            if "ERROR" in line:
                return line

In [ ]:
def slow_square(numbers):
    for n in numbers:
        print(f"Calculating square of {n}...")
        return n * n

my_numbers = [1, 2, 3]

# looping through this function will raise an error because once u returned one thing th e function is removed
slow_square(my_numbers)
# u can create a list and append to it or yield

Calculating square of 1...


1

In [42]:
def slow_square(numbers):
    for n in numbers:
        print(f"Calculating square of {n}...")
        yield n * n

my_numbers = [1, 2, 3]

In [43]:
#eager #bad for memory
# # Square brackets []
squares_list = [x for x in slow_square(my_numbers)]
print("Starting the loop!")
for s in squares_list:
    print(f"Result: {s}")

Calculating square of 1...
Calculating square of 2...
Calculating square of 3...
Starting the loop!
Result: 1
Result: 4
Result: 9


In [44]:
#lazy
# Parentheses ()
squares_gen = (x for x in slow_square(my_numbers))
print("Starting the loop!")
for s in squares_gen:
    print(f"Result: {s}")

Starting the loop!
Calculating square of 1...
Result: 1
Calculating square of 2...
Result: 4
Calculating square of 3...
Result: 9


In [47]:
import itertools

l = list(itertools.accumulate(range(1,11)))
l

[1, 3, 6, 10, 15, 21, 28, 36, 45, 55]

In [49]:
list(itertools.zip_longest(range(1,8), l))

[(1, 1),
 (2, 3),
 (3, 6),
 (4, 10),
 (5, 15),
 (6, 21),
 (7, 28),
 (None, 36),
 (None, 45),
 (None, 55)]

In [55]:
animals = ['duck', 'eagle', 'rat', 'giraffe', 'cat',
'bat', 'dolphin', 'shark', 'lion']
# animals.sort(key=len)
# animals
for length, name in itertools.groupby(animals, len):
    print(length, list(name))

4 ['duck']
5 ['eagle']
3 ['rat']
7 ['giraffe']
3 ['cat', 'bat']
7 ['dolphin']
5 ['shark']
4 ['lion']


In [ ]:
a,b = itertools.tee('abd')
print(next(a))
print(next(b))

list(zip(*itertools.tee('abd'))) #The * unpacks the two iterators into separate arguments for zip.

a
a


[('a', 'a'), ('b', 'b'), ('d', 'd')]

In [70]:
print(all([0,0,0]))
print(all([]))
print(any([]))

False
True
False


In [73]:
g = (n for n in [1,2,0,3])
print(all(g))
next(g)

False


3

 coroutine is really a generator function,
created with the yield keyword in its body. And a coroutine object is physically a
generator object. Despite sharing the same underlying implementation in C, the use
cases of generators and coroutines in Python are so different

generators can actually be a two-way street. You can pause a generator, compute something, and then push new data into the paused generator using a method called .send(). When a generator is used this way, it is called a coroutine.

In [84]:
def court():
    print('start')

    status = yield 'ready'
    print('coroutine received:', status)

c = court()
next(c)
try:
    c.send('hello')
except StopIteration:
    pass





start
coroutine received: hello


In [85]:
def court():
    print('start')
    while True:
        status = yield 'ready'
        print('coroutine received:', status)

c = court()
next(c)

c.send('hello')
c.close()




start
coroutine received: hello


In [89]:
def kitchen(order_name):
    """CHILD COROUTINE: Does the actual work."""
    print(f"  [Kitchen] Received ticket for: {order_name}")
    
    # 1. Yield out a status, pause, and wait for the customer to send a message
    message_from_customer = yield f"Prep started for {order_name}..."
    print(f"  [Kitchen] Customer yelled: '{message_from_customer}'")
    
    # 2. Yield out another status, pause again
    yield f"Cooking the {order_name}..."
    
    # 3. The coroutine is done. Return the final product!
    # (Remember: standard loops crash here, but 'yield from' catches this beautifully)
    return f"A perfectly cooked {order_name}"


def waiter():
    """PARENT COROUTINE: Delegates to the kitchen using 'yield from'."""
    print("[Waiter] Welcome! I'll send your order to the back.")
    
    # --- THE MAGIC PIPE ---
    # 1. Any 'yield' from the kitchen flows right through to the customer.
    # 2. Any '.send()' from the customer flows right through to the kitchen.
    # 3. When the kitchen hits 'return', 'yield from' catches the StopIteration 
    #    and hands us the return value!
    final_dish = yield from kitchen("Steak")
    
    print(f"[Waiter] Ah, the kitchen handed me: {final_dish}")
    
    # The waiter returns the final experience
    return "Meal complete! Here is your bill."


# ==========================================
# MAIN PROGRAM (The Customer)
# ==========================================

print("--- 1. Customer arrives ---")
my_experience = waiter()

print("\n--- 2. Customer asks the Waiter to start ---")
# Prime the parent, which primes the child
status1 = next(my_experience) 
print(f"(Customer hears): {status1}")

print("\n--- 3. Customer sends a message THROUGH the Waiter to the Kitchen ---")
# We send data IN. It bypasses the Waiter entirely and lands in the Kitchen's 'yield'!
status2 = my_experience.send("Hurry up, I'm hungry!")
print(f"(Customer hears): {status2}")

print("\n--- 4. Customer waits for the food ---")
# We send one last push. The kitchen finishes and hits 'return'.
# The waiter catches the food, prints his line, and then hits his own 'return'.
try:
    my_experience.send("Is it done yet?")
except StopIteration as exc:
    # We catch the Waiter's final return value from the StopIteration briefcase
    final_greeting = exc.value
    print(f"(Customer hears): {final_greeting}")

--- 1. Customer arrives ---

--- 2. Customer asks the Waiter to start ---
[Waiter] Welcome! I'll send your order to the back.
  [Kitchen] Received ticket for: Steak
(Customer hears): Prep started for Steak...

--- 3. Customer sends a message THROUGH the Waiter to the Kitchen ---
  [Kitchen] Customer yelled: 'Hurry up, I'm hungry!'
(Customer hears): Cooking the Steak...

--- 4. Customer waits for the food ---
[Waiter] Ah, the kitchen handed me: A perfectly cooked Steak
(Customer hears): Meal complete! Here is your bill.


The "Switchboard Operator" Analogy
When the Waiter executes the line yield from kitchen("Steak"), the Waiter essentially becomes an old-school telephone switchboard operator.

The Waiter takes the Customer's phone line.

The Waiter plugs it directly into the Kitchen's phone line.

The Waiter takes off their headset and goes to sleep.

While yield from is running, the Waiter is completely paused. The Waiter cannot see, hear, or interact with the data flowing back and forth. The yield from command has created a direct tunnel bypassing the Waiter's logic entirely.

When the Customer calls .send("Hurry up!"), Python sees the tunnel, skips the Waiter's code, and injects the message directly into the Kitchen's active yield statement.